# Sequential Solution Methods

Compare exact dynamic programming and Greedy on the saved one- and two-dimensional datasets. Budgets are $N_{\min}^{\mathrm{seq}}$ and $N_{\mathrm{greedy}}^{\mathrm{seq}}$.

Notebooks 01, 02, and 03 run independently from the inputs. After all three full runs, use `final_figures.ipynb` to redraw their saved results.


In [ ]:
from pathlib import Path
from collections import defaultdict
import csv
import math
import sys

from IPython.display import Image, Markdown, display

working_directory = Path.cwd().resolve()
PROJECT_DIRECTORY = (
    working_directory / "problem-1-zero-error"
    if (working_directory / "problem-1-zero-error").is_dir()
    else working_directory
)
SOURCE_DIRECTORY = PROJECT_DIRECTORY / "src"
SCRIPT_DIRECTORY = PROJECT_DIRECTORY / "scripts"
for directory in (SOURCE_DIRECTORY, SCRIPT_DIRECTORY):
    if str(directory) not in sys.path:
        sys.path.insert(0, str(directory))

local_modules = (
    "run_config", "optimal", "greedy", "phase_data", "field_models", "eval",
    "benchmark_analysis", "generate_inputs", "generate_benchmarks",
    "evaluate_benchmarks",
)
for module_name in local_modules:
    sys.modules.pop(module_name, None)

from evaluate_benchmarks import DATASETS, run_benchmark_evaluation
from benchmark_analysis import family_excess_table, guarantee_table, heatmap_table

print("Setup complete.")
print("Working directory:", working_directory)
print("Project directory:", PROJECT_DIRECTORY)
print("Source directory:", SOURCE_DIRECTORY)
print("Script directory:", SCRIPT_DIRECTORY)
print("Datasets queued:", ", ".join(DATASETS))


## Run configuration

Set `RUN_MODE` to `"test"` (132 cases: all families and K, M=3 or 6, replicate 1) or `"full"` (39,600 cases). Run all cells after changing it. Test and full outputs are separate; final figures use full results.

`REPEATS` controls timing repeats; `BOOTSTRAP_ITERATIONS` controls confidence intervals. Keep `GENERATE_MISSING_INPUTS = False` to use saved inputs.


In [ ]:
RUN_MODE = "full"  # "test" or "full"

if RUN_MODE not in ("test", "full"):
    raise ValueError('RUN_MODE must be "test" or "full"')
print("Run mode:", RUN_MODE)
print("Expected standard benchmark cases:", 132 if RUN_MODE == "test" else 39_600)

REPEATS = 1
BOOTSTRAP_ITERATIONS = 200 if RUN_MODE == "test" else 2_000
GENERATE_MISSING_INPUTS = False
print("Bootstrap iterations:", BOOTSTRAP_ITERATIONS)


## Run evaluation

Evaluate both datasets and save tables and figures.


In [ ]:
print("Starting zero-error benchmark evaluation.")
print("The evaluator will print one progress block per dataset.")
analysis = run_benchmark_evaluation(
    run_mode=RUN_MODE,
    repeats=REPEATS,
    bootstrap_iterations=BOOTSTRAP_ITERATIONS,
    generate_missing=GENERATE_MISSING_INPUTS,
)
print("Evaluation complete.")
print("Analyzed rows:", len(analysis["rows"]))
print("Analysis output directory:", analysis["output_directory"])

ANALYSIS_FIGURES = ("families", "guarantee", "heatmaps", "allocation_match")
if set(analysis["figures"]) != set(ANALYSIS_FIGURES):
    raise RuntimeError("The expected analysis figures were not created.")
print("Verified analysis figures:", ", ".join(ANALYSIS_FIGURES))


## Numerical summary

Results are shown overall and by dimension. Quality is measured by $N_{\mathrm{greedy}}^{\mathrm{seq}}/N_{\min}^{\mathrm{seq}}$.


In [ ]:
def summary_table(rows):
    header = (
        "| Group | Cases | Mean ratio | 95% CI | Budget-match rate | "
        "95th percentile | Maximum |\n"
        "|---|---:|---:|---:|---:|---:|---:|"
    )
    body = []
    for row in rows:
        body.append(
            f"| {row['group_value']} | {row['case_count']} | "
            f"{row['mean_greedy_ratio']:.3f} | "
            f"[{row['mean_greedy_ratio_ci95_low']:.3f}, "
            f"{row['mean_greedy_ratio_ci95_high']:.3f}] | "
            f"{100 * row['exact_match_rate']:.1f}% | "
            f"{row['p95_greedy_ratio']:.3f} | "
            f"{row['max_greedy_ratio']:.3f} |"
        )
    return header + "\n" + "\n".join(body)

summary_rows = [analysis["overall_summary"], *analysis["dimension_summaries"]]
print(f"Displaying {len(summary_rows)} summary rows.")
display(Markdown(summary_table(summary_rows)))


## Figure data

Numerical tables behind the comparison plots.


In [ ]:
def markdown_table(rows, columns, formats):
    header = "| " + " | ".join(label for label, _ in columns) + " |"
    separator = "|" + "|".join("---:" if formats.get(key) else "---" for _, key in columns) + "|"
    body = []
    for row in rows:
        values = []
        for _, key in columns:
            formatter = formats.get(key)
            value = row[key]
            values.append(formatter(value) if formatter else str(value))
        body.append("| " + " | ".join(values) + " |")
    return "\n".join([header, separator, *body])

heatmap_rows = heatmap_table(analysis["rows"])
print(f"Displaying quality heatmap table with {len(heatmap_rows)} rows.")
display(Markdown("### Quality Heatmap Values"))
display(Markdown(markdown_table(
    heatmap_rows,
    (
        ("Dimension", "dimension"),
        ("K", "K"),
        ("M", "M"),
        ("Mean additional sensors used by Greedy (%)", "mean_additional_sensors_percent"),
    ),
    {"K": lambda value: f"{int(value)}", "M": lambda value: f"{int(value)}", "mean_additional_sensors_percent": lambda value: f"{float(value):.2f}"},
)))

family_rows = family_excess_table(
    analysis["rows"],
    bootstrap_iterations=BOOTSTRAP_ITERATIONS,
)
print(f"Displaying field-family figure table with {len(family_rows)} rows.")
display(Markdown("### Field-Family Bar-Chart Values"))
display(Markdown(markdown_table(
    family_rows,
    (
        ("Dimension", "dimension"),
        ("Field family", "field_family"),
        ("K", "K"),
        ("Mean additional sensors used by Greedy (%)", "mean_additional_sensors_percent"),
        ("95% CI low", "ci95_low"),
        ("95% CI high", "ci95_high"),
    ),
    {
        "K": lambda value: f"{int(value)}",
        "mean_additional_sensors_percent": lambda value: f"{float(value):.2f}",
        "ci95_low": lambda value: f"{float(value):.2f}",
        "ci95_high": lambda value: f"{float(value):.2f}",
    },
)))

guarantee_rows = guarantee_table(analysis["rows"])
print(f"Displaying approximation-guarantee table with {len(guarantee_rows)} rows.")
display(Markdown("### Approximation-Guarantee Figure Values"))
display(Markdown(markdown_table(
    guarantee_rows,
    (
        ("Dimension", "dimension"),
        ("Field family", "field_family"),
        ("K", "K"),
        ("Maximum worst-case allowance used (%)", "maximum_worst_case_allowance_used_percent"),
    ),
    {"K": lambda value: f"{int(value)}", "maximum_worst_case_allowance_used_percent": lambda value: f"{float(value):.2f}"},
)))


## Method comparison

Sensor excess by field family, worst-case allowance used, and K-by-M quality heatmaps.


In [ ]:
PUBLICATION_FIGURES = ("families", "guarantee", "heatmaps")
plot_titles = {
    "families": "Sensor excess by field family",
    "guarantee": "Approximation guarantee comparison",
    "heatmaps": "Quality heatmaps",
}
plot_files = {
    "families": "benchmark_family_excess.png",
    "guarantee": "benchmark_guarantee_comparison.png",
    "heatmaps": "benchmark_quality_heatmaps.png",
}
figure_directory = Path(analysis["output_directory"]) / "figures"
print("Displaying solution-method figures from:", figure_directory)
for figure_key in PUBLICATION_FIGURES:
    figure_path = figure_directory / plot_files[figure_key]
    print(f"Displaying {figure_key}: {figure_path}")
    display(Markdown(f"### {plot_titles[figure_key]}"))
    display(Image(filename=str(figure_path)))


## Allocation agreement

Families and location counts are pooled at each K. “Same allocation” compares sensor counts at every location; “same total budget” compares their sums. Different allocations can both be optimal, so agreement is measured against the particular DP solution returned.


In [ ]:
match_rows = analysis["allocation_match_table"]
display(Markdown(markdown_table(
    match_rows,
    (("Dimension", "dimension"), ("K", "K"), ("Cases", "case_count"),
     ("Allocation matches", "allocation_matches"), ("Budget matches", "budget_matches"),
     ("Same allocation (%)", "allocation_match_percent"),
     ("Same total budget (%)", "budget_match_percent")),
    {"K": lambda x: str(int(x)), "case_count": lambda x: str(int(x)),
     "allocation_matches": lambda x: str(int(x)), "budget_matches": lambda x: str(int(x)),
     "allocation_match_percent": lambda x: f"{float(x):.2f}",
     "budget_match_percent": lambda x: f"{float(x):.2f}"},
)))
match_figure_path = Path(analysis["output_directory"]) / "figures" / "allocation_match.png"
display(Image(filename=str(match_figure_path)))
print("Saved table:", Path(analysis["output_directory"]) / "tables" / "allocation_match.csv")


## Saved outputs

Results go to `results/methods/{test,full}/`, with combined `tables/` and `figures/` and per-dataset details in `1d/tables/` and `2d/tables/`.
